### Importing the necessary libraries

In [ ]:
import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Implementing text search with sci-kit learn

In [ ]:
# we will start off by converting to json the FAQs from the various courses from DataTalks - this had already
# been parsed by the course instructor 
# this will be used as the knowledge/data base by the search engine that we should be building shortly

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [ ]:
documents[2]

In [ ]:
# converting to pandas dataframe

df = pd.DataFrame(documents, columns=['course', 'section', 'question', 'text'])
df.head()

In [ ]:
# the next step is to vectorise the text column but to understand how vectorisation works
# we can look at the following example - we start off with a list of texts called sample

sample = [
    "Course starts on 15th Jan 2024",
    "Prerequisites listed on GitHub",
    "Submit homeworks after start date",
    "Registration not required for participation",
    "Setup Google Cloud and Python before course"
]

In [ ]:
# next we initialise the object so that we can convert it into a encoded vector

cv = CountVectorizer(stop_words='english')
X = cv.fit_transform(sample)

names = cv.get_feature_names_out()

# this is what is called a bag of words - there is no order
df_docs = pd.DataFrame(X.toarray(), columns=names).T
df_docs  # following is an illustration

In [ ]:
# this is what we need - but not the best as there is no importance

cv = CountVectorizer(stop_words='english', min_df=5)
X = cv.fit_transform(df.text) # sparse matrix

In [ ]:
# hence we use TF-IDF (Term Frequency-Inverse Document Frequency)
# to give less importance to words that appear more frequently

tf = TfidfVectorizer(stop_words='english', min_df=5)
X = tf.fit_transform(df.text) # sparse matrix

In [ ]:
X

In [ ]:
# now to represent the query in the same vector space - i.e. using the same vectorizer

query = "I just discovered the course, is it too late to join?"

q = tf.transform([query])
q.toarray()

In [ ]:
# we need to take the weight of the term in the query (weighted as its a tfid vectoriser)
# then we need to multiply that by the weight in the document, followed by the sum across all the
# matching terms to give a measure of similarity - i.e. how similar is the query for the document.
# If we repeat the process we can score all the documents, and rank them based on relevance score.

X.dot(q.T).todense()

In [ ]:
# so essentially what we were trying to derive thus far is actually cosine similarity
# cosine similarity method provides a number to describe the similarity between a 
# query and the document like below - to discuss more on cosine_similarity

score = cosine_similarity(X, q).flatten()

In [ ]:
# array of indexes of the highest scores in SCORES - highest scores at bottom

np.argsort(score)[-5:]

In [ ]:
# document 22 is the highest matching document to the query

df.iloc[22].text

In [ ]:
# we can repeat the steps for fields other than text

fields = ['section', 'question', 'text']

matrices = {}
vectorizers = {}

for f in fields:
    cv = TfidfVectorizer(stop_words='english', min_df=5)
    X = cv.fit_transform(df[f])
    matrices[f] = X
    vectorizers[f] = cv

In [ ]:
matrices

In [ ]:
vectorizers

In [ ]:
n = len(df)

score = np.zeros(n)

boosts = {
    'question' : 3  # question field has more relevance than other - hence 3 times 
}

for f in fields:
    q = vectorizers[f].transform([query])
    X = matrices[f]

    f_score = cosine_similarity(X,q).flatten()
    
    boost = boosts.get(f, 1)

    score += boost * f_score

In [ ]:
# its important to add filters so as to retrive documents that are most relevant

filters = {
    'course' : 'data-engineering-zoomcamp'
}

for field, value in filters.items():
    mask = (df[field] == value).astype(int).values # converting boolean to int
    score = score * mask

In [ ]:
idx = np.argsort(-score)[:10]
results = df.iloc[idx]
results.to_dict(orient='records')

In [ ]:
results

### Putting it all together - Let's create a class for us to use

In [ ]:
class TextSearch:

    def __init__(self, text_fields):
        self.text_fields = text_fields
        self.matrices = {}
        self.vectorizers = {}

    def fit(self, records, vectorizer_params={}):
        self.df = pd.DataFrame(records)

        for f in self.text_fields:
            cv = TfidfVectorizer(**vectorizer_params)
            X = cv.fit_transform(self.df[f])
            self.matrices[f] = X
            self.vectorizers[f] = cv

    def search(self, query, n_results=10, boost={}, filters={}):
        score = np.zeros(len(self.df))

        for f in self.text_fields:
            b = boost.get(f, 1.0)
            q = self.vectorizers[f].transform([query])
            s = cosine_similarity(self.matrices[f], q).flatten()
            score = score + b * s

        for field, value in filters.items():
            mask = (self.df[field] == value).values
            score = score * mask

        idx = np.argsort(-score)[:n_results]
        results = self.df.iloc[idx]
        return results.to_dict(orient='records')

In [ ]:
# using it

index = TextSearch(
    text_fields=['section', 'question', 'text']
)
index.fit(documents)

index.search(
    query='I just singned up. Is it too late to join the course?',
    n_results=5,
    boost={'question': 3.0},
    filters={'course': 'data-engineering-zoomcamp'}
)